# Chat fine-tuning (SFT) of the pretrained 100M model - Kaggle **T4 x2**
Takes the base model from the pretraining run, adds ChatML tokens (`<|im_start|>`, `<|im_end|>`), and fine-tunes on **HuggingFaceTB/smol-smoltalk** (Apache-2.0, built for <1B-parameter chat models) with loss **only on assistant replies**.

**Before running:** Add Input -> attach the **output of your pretraining notebook** (must contain `final_model/`). Accelerator: **GPU T4 x2**, Internet **ON**. Run with *Save & Run All* or cell by cell.

Set `QUICK_TEST = True` in the first cell for a cheap ~45-min dry run (6k conversations) before the full run.

Result: `/kaggle/working/chat_model` (model + tokenizer with chat template + `chat.py`).

In [ ]:
import subprocess, sys, os, json
from pathlib import Path

# ====== settings you may change ======
WALL_LIMIT_H = 5.0      # total session hours for SFT (data prep + training + save). Longer = more epochs. Safe max: 10.5
QUICK_TEST = False     # True = ~45-min end-to-end dry run on 6k conversations (checks prep/train/save/chat cheaply)
# =====================================
EXTRA_ENV = {}
if QUICK_TEST:
    WALL_LIMIT_H = 0.75
    EXTRA_ENV = {"SFT_TAG": "_quick", "MAX_CONVS": "6000", "VAL_CONVS": "500"}
TAG = EXTRA_ENV.get("SFT_TAG", "")
CHAT_DIR = "/kaggle/working/chat_model" + TAG
print("QUICK TEST" if QUICK_TEST else "FULL RUN", "| wall limit", WALL_LIMIT_H, "h")

print(sys.version)
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
need = [m for m in ("transformers", "datasets", "tokenizers", "huggingface_hub", "psutil", "safetensors")
        if __import__("importlib.util").util.find_spec(m) is None]
if need: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *need])

# ---- locate the pretrained model (output of the pretraining notebook, added via "Add Input") ----
cands = []
for root, dirs, files in os.walk("/kaggle/input"):
    if root.count("/") - 2 >= 6: dirs[:] = []
    if "config.json" in files and "tokenizer.json" in files and ("model.safetensors" in files or "pytorch_model.bin" in files):
        cands.append(root)
if Path("/kaggle/working/final_model/config.json").exists(): cands.append("/kaggle/working/final_model")   # same-session fallback
cands.sort(key=lambda p: (0 if p.rstrip("/").endswith("final_model") else 1, len(p)))
if not cands:
    raise FileNotFoundError("No pretrained model found. Use 'Add Input' and attach the OUTPUT of your pretraining notebook "
                            "(it must contain final_model/config.json + model.safetensors + tokenizer.json).")
PRETRAINED_DIR = cands[0]
os.environ["PRETRAINED_DIR"] = PRETRAINED_DIR
print("Pretrained model:", PRETRAINED_DIR)
if len(cands) > 1: print("  (other candidates:", cands[1:], ")")
summ = Path(PRETRAINED_DIR) / "training_summary.json"
if summ.exists():
    s = json.loads(summ.read_text()); print(f"  pretraining: {s.get('params_M', 0):.0f}M params, {s.get('tokens_seen', 0)/1e6:.0f}M tokens, final val loss {s.get('final_val_loss', float('nan')):.3f}")

In [ ]:
%%writefile /kaggle/working/sft_pipeline.py
"""
Chat fine-tuning (SFT) of the pretrained Llama-style base model, Kaggle T4 x2 (DDP, fp16 AMP).

  python sft_pipeline.py prep                                      # chat tokenizer + tokenized conversations (CPU)
  torchrun --standalone --nproc_per_node=2 sft_pipeline.py train   # DDP fine-tuning on both GPUs

Input : the pretraining output folder (final_model: weights + tokenizer), auto-discovered or via PRETRAINED_DIR
Data  : HuggingFaceTB/smol-smoltalk (Apache-2.0, built for <1B-parameter chat models)
Format: ChatML  <|im_start|>role\\ncontent<|im_end|>\\n ; loss only on assistant replies (+ <|im_end|>)
Output: /kaggle/working/chat_model  (model + tokenizer w/ chat template + generation_config + chat.py)
"""
import os, sys, time, math, json, queue, threading, shutil, contextlib
from datetime import timedelta
from pathlib import Path
import numpy as np
import torch
import torch.distributed as dist

SMOKE = os.environ.get("SMOKE", "0") == "1"
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "60")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

if os.environ.get("SESSION_T0"):
    T0 = float(os.environ["SESSION_T0"])
else:
    try:
        import psutil; T0 = psutil.Process().create_time()
    except Exception:
        T0 = time.time()

RANK = int(os.environ.get("RANK", 0))
IS_MAIN = RANK == 0
ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("./work")
TAG = os.environ.get("SFT_TAG", "")              # "_quick" for the dry run, so it never mixes with the full run
DATA = (Path("/kaggle/temp") if Path("/kaggle/temp").exists() else WORK / "data") / f"sft_data{TAG}"
CHAT_TOK = DATA / "chat_tokenizer"
CKPT = WORK / f"sft_ckpt{TAG}"
OUT = WORK / f"chat_model{TAG}"

# ---- time budget ----
WALL_LIMIT_H = float(os.environ.get("WALL_LIMIT_H", "5.0"))   # whole session (prep + train + save). Raise for longer SFT
FINAL_RESERVE_MIN = 12

# ---- data ----
DATASET = "HuggingFaceTB/smol-smoltalk"
MAX_CONVS, VAL_CONVS, MAX_LEN = int(os.environ.get("MAX_CONVS", "300000")), int(os.environ.get("VAL_CONVS", "2000")), 1024
BOS, PAD, IM_START, IM_END = "<|endoftext|>", "<|pad|>", "<|im_start|>", "<|im_end|>"
ROLES = {"system", "user", "assistant"}
CHAT_TEMPLATE = (
    "{{ bos_token }}"
    "{% for m in messages %}"
    "{{ '<|im_start|>' + m['role'] + '\\n' + m['content'] | trim + '<|im_end|>' + '\\n' }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|>assistant\\n' }}{% endif %}"
)

# ---- optimisation ----
TOKENS_PER_MB = int(os.environ.get("TOKENS_PER_MB", "4096"))  # padded tokens per GPU micro-batch (same as pretraining's 4x1024)
GLOBAL_BATCH_TOKENS = 32_768
PEAK_LR, MIN_LR_FRAC = 3e-4, 0.1
WARMUP_STEPS = 100
WEIGHT_DECAY, BETAS, CLIP = 0.01, (0.9, 0.95), 1.0
MAX_EPOCHS = 3.0                 # LR schedule ends at whichever comes first: time budget or this many epochs
LOG_EVERY, EVAL_EVERY, SAMPLE_EVERY, EVAL_BATCHES = 25, 250, 500, 40
CKPT_EVERY_MIN = 30
COMM_TIMEOUT_MIN = 15
SEED = 1337
TRAIN_SECONDS_OVERRIDE = None

if SMOKE:
    MAX_CONVS, VAL_CONVS, MAX_LEN = 300, 20, 128
    TOKENS_PER_MB, GLOBAL_BATCH_TOKENS, WARMUP_STEPS = 512, 1024, 3
    LOG_EVERY, EVAL_EVERY, SAMPLE_EVERY, EVAL_BATCHES = 3, 10, 20, 4
    TRAIN_SECONDS_OVERRIDE = 40
    DATA = WORK / "sft_smoke" / "data"; CHAT_TOK = DATA / "chat_tokenizer"
    CKPT, OUT = WORK / "sft_smoke" / "ckpt", WORK / "sft_smoke" / "out"

IDS_BIN, MASK_BIN, LENS_NPY, META = DATA / "ids.bin", DATA / "mask.bin", DATA / "lens.npy", DATA / "meta.json"
for d in (WORK, DATA): d.mkdir(parents=True, exist_ok=True)

def wall_elapsed(): return time.time() - T0
def log(*a):
    if IS_MAIN: print(f"[{wall_elapsed()/3600:5.2f}h]", *a, flush=True)

# ------------------------------------------------------------------ chat format
def clean_text(s):
    for t in (BOS, PAD, IM_START, IM_END): s = s.replace(t, "")
    return s

def render(messages, add_generation_prompt=False):
    """ChatML text + char spans of the trainable parts (assistant content + <|im_end|>).
    Must stay byte-identical to CHAT_TEMPLATE (checked at the end of training)."""
    text, spans = BOS, []
    for m in messages:
        role, content = m["role"], m["content"].strip()
        text += f"{IM_START}{role}\n"
        if role == "assistant":
            s = len(text); text += content + IM_END; spans.append((s, len(text))); text += "\n"
        else:
            text += content + IM_END + "\n"
    if add_generation_prompt: text += f"{IM_START}assistant\n"
    return text, spans

def normalise(messages):
    """Keep system/user/assistant turns with text, end on an assistant turn; None if unusable."""
    out = []
    for m in messages:
        role, content = m.get("role"), m.get("content")
        if role not in ROLES or not isinstance(content, str): return None
        content = clean_text(content).strip()
        if not content: return None
        out.append({"role": role, "content": content})
    while out and out[-1]["role"] != "assistant": out.pop()
    if not any(m["role"] == "user" for m in out) or not any(m["role"] == "assistant" for m in out): return None
    return out

# ------------------------------------------------------------------ locating the pretrained model
def _scan(base, maxdepth=6):
    found = []
    base = str(base)
    if not os.path.isdir(base): return found
    b0 = base.rstrip("/").count("/")
    for root, dirs, files in os.walk(base):
        if root.count("/") - b0 >= maxdepth: dirs[:] = []
        if "config.json" in files and "tokenizer.json" in files and \
           ("model.safetensors" in files or "pytorch_model.bin" in files):
            found.append(root)
    return found

def find_pretrained():
    env = os.environ.get("PRETRAINED_DIR")
    if env and (Path(env) / "config.json").exists(): return env
    cands = _scan("/kaggle/input") + _scan(WORK / "final_model", 2)
    cands = [c for c in cands if "chat_model" not in c]
    cands.sort(key=lambda p: (0 if p.rstrip("/").endswith("final_model") else 1, len(p)))
    if not cands:
        raise FileNotFoundError(
            "Pretrained model not found. Add your pretraining notebook's output as an input "
            "(Add Input -> Notebook Output / Dataset) so that .../final_model/config.json exists, "
            "or set PRETRAINED_DIR.")
    return cands[0]

# ------------------------------------------------------------------ data
def hf_login():
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF_TOKEN")
        if t:
            from huggingface_hub import login
            login(token=t); log("Logged in to Hugging Face")
    except Exception:
        log("No HF_TOKEN secret - continuing anonymously (smol-smoltalk is public).")

def conv_stream():
    if SMOKE:
        import random
        rng = random.Random(0); words = ["alpha", "beta", "gamma", "delta", "tree", "river", "model", "answer", "hello"]
        def sent(a, b): return " ".join(rng.choice(words) for _ in range(rng.randint(a, b)))
        while True:
            msgs = [{"role": "system", "content": sent(3, 8)}] if rng.random() < 0.3 else []
            for _ in range(rng.randint(1, 3)):
                msgs += [{"role": "user", "content": sent(4, 25)}, {"role": "assistant", "content": sent(5, 60)}]
            yield msgs
    from datasets import load_dataset
    seen, fails = 0, 0
    while True:
        try:
            ds = load_dataset(DATASET, split="train", streaming=True)
            if seen: ds = ds.skip(seen)
            for ex in ds:
                seen += 1; fails = 0
                yield ex["messages"]
            return
        except Exception as e:
            fails += 1
            if fails > 8: raise
            wait = min(120, 5 * 2 ** fails)
            log(f"Stream error ({type(e).__name__}: {str(e)[:120]}); retry {fails}/8 in {wait}s")
            time.sleep(wait)

def make_chat_tokenizer():
    from transformers import AutoTokenizer
    if (CHAT_TOK / "tokenizer.json").exists():
        return AutoTokenizer.from_pretrained(str(CHAT_TOK))
    pre = find_pretrained()
    log("Pretrained model:", pre)
    tok = AutoTokenizer.from_pretrained(pre)
    old = len(tok)
    assert tok.convert_tokens_to_ids(BOS) is not None and tok.convert_tokens_to_ids(PAD) is not None, "unexpected base tokenizer"
    tok.add_special_tokens({"additional_special_tokens": [IM_START, IM_END]})
    tok.eos_token = IM_END                         # generation stops at <|im_end|>
    tok.chat_template = CHAT_TEMPLATE
    CHAT_TOK.mkdir(parents=True, exist_ok=True)
    tok.save_pretrained(str(CHAT_TOK))
    log(f"Chat tokenizer: vocab {old} -> {len(tok)} | {IM_START}={tok.convert_tokens_to_ids(IM_START)} "
        f"{IM_END}={tok.convert_tokens_to_ids(IM_END)} bos={tok.convert_tokens_to_ids(BOS)} pad={tok.convert_tokens_to_ids(PAD)}")
    return tok

def build_data():
    if META.exists() and IDS_BIN.exists() and MASK_BIN.exists() and LENS_NPY.exists():
        log("Tokenized conversations already exist:", json.loads(META.read_text())); return
    tok = make_chat_tokenizer()
    assert len(tok) <= 65535
    log(f"Tokenizing up to {MAX_CONVS + VAL_CONVS} conversations (max {MAX_LEN} tokens each) ...")
    lens, stats = [], {"seen": 0, "bad": 0, "too_long": 0, "no_target": 0}
    want = MAX_CONVS + VAL_CONVS
    f_ids, f_mask = open(IDS_BIN, "wb"), open(MASK_BIN, "wb")
    pending, t, nflush = [], time.time(), 0

    def flush():
        nonlocal pending
        texts = [p[0] for p in pending]
        enc = tok(texts, add_special_tokens=False, return_offsets_mapping=True)
        for (text, spans), ids, offs in zip(pending, enc["input_ids"], enc["offset_mapping"]):
            if len(lens) >= want: break
            n = len(ids)
            if n > MAX_LEN or n < 4: stats["too_long"] += 1; continue
            offs = np.asarray(offs, dtype=np.int64).reshape(-1, 2)
            ss = np.asarray([s for s, _ in spans], dtype=np.int64); se = np.asarray([e for _, e in spans], dtype=np.int64)
            k = np.searchsorted(ss, offs[:, 0], side="right") - 1
            kc = np.clip(k, 0, None)
            inside = (k >= 0) & (offs[:, 1] <= se[kc])          # token lies fully inside an assistant span
            if not inside.any(): stats["no_target"] += 1; continue
            f_ids.write(np.asarray(ids, dtype=np.uint16).tobytes())
            f_mask.write(inside.astype(np.uint8).tobytes())
            lens.append(n)
        pending = []

    for msgs in conv_stream():
        stats["seen"] += 1
        m = normalise(msgs)
        if m is None: stats["bad"] += 1; continue
        pending.append(render(m))
        if len(pending) == 512:
            flush(); nflush += 1
            if nflush % 40 == 0:
                log(f"  {len(lens):,} convs kept | seen {stats['seen']:,} | {time.time()-t:.0f}s")
        if len(lens) >= want: break
    if pending and len(lens) < want: flush()
    f_ids.close(); f_mask.close()
    lens = np.asarray(lens, dtype=np.int32)
    np.save(LENS_NPY, lens)
    META.write_text(json.dumps({"n_convs": int(len(lens)), "n_val": VAL_CONVS, "total_tokens": int(lens.sum()), **stats}))
    log(f"Done: {len(lens):,} convs, {int(lens.sum())/1e6:.1f}M tokens | {stats} | {(time.time()-t)/60:.1f} min")

class ConvData:
    """Tokenized conversations on disk (numpy memmaps). Torch-free so it can be unit-tested."""
    def __init__(self):
        self.ids = np.memmap(IDS_BIN, dtype=np.uint16, mode="r")
        self.mask = np.memmap(MASK_BIN, dtype=np.uint8, mode="r")
        self.lens = np.load(LENS_NPY).astype(np.int64)
        self.offs = np.concatenate([[0], np.cumsum(self.lens)[:-1]]).astype(np.int64)
        meta = json.loads(META.read_text())
        self.n_val = int(meta["n_val"])
        assert int(self.offs[-1] + self.lens[-1]) == len(self.ids) == len(self.mask), "corrupt token files - delete sft_data and rerun prep"
        assert len(self.lens) > self.n_val + 100, "too few conversations"
        self.val_idx = np.arange(self.n_val)
        self.train_idx = np.arange(self.n_val, len(self.lens))
        self.train_tokens = int(self.lens[self.train_idx].sum())

    def collate(self, idx, pad_id):
        n, L = len(idx), int(self.lens[idx].max())
        x = np.full((n, L), pad_id, np.int64); y = np.full((n, L), -100, np.int64); am = np.zeros((n, L), np.int64)
        for r, i in enumerate(idx):
            o, l = int(self.offs[i]), int(self.lens[i])
            ids = self.ids[o:o + l].astype(np.int64); m = self.mask[o:o + l].astype(bool)
            x[r, :l] = ids; y[r, :l] = np.where(m, ids, -100); am[r, :l] = 1
        return x, y, am

    def make_batches(self, idx_sorted, budget):
        """Greedy length-bucketed micro-batches: n * max_len <= budget (idx_sorted ascending by length)."""
        batches, i = [], 0
        while i < len(idx_sorted):
            j = i
            while j < len(idx_sorted) and (j - i + 1) * int(self.lens[idx_sorted[j]]) <= budget: j += 1
            j = max(j, i + 1)
            batches.append(idx_sorted[i:j]); i = j
        return batches

    def train_batches(self, seed, budget, chunk=20000):
        rng = np.random.default_rng(seed)
        while True:
            order = rng.permutation(self.train_idx)
            for c in range(0, len(order), chunk):
                part = order[c:c + chunk]
                part = part[np.argsort(self.lens[part], kind="stable")]
                bs = self.make_batches(part, budget)
                for k in rng.permutation(len(bs)): yield bs[k]

class Loader:
    """Background thread: builds padded batches, pins them, hands them over via a queue."""
    def __init__(self, data, seed, budget, pad_id, device_index=None, prefetch=6):
        self.q = queue.Queue(maxsize=prefetch)
        threading.Thread(target=self._work, args=(data, seed, budget, pad_id, device_index), daemon=True).start()
    def _work(self, data, seed, budget, pad_id, device_index):
        try:
            pin = torch.cuda.is_available()
            if pin and device_index is not None: torch.cuda.set_device(device_index)
            for idx in data.train_batches(seed, budget):
                t = tuple(torch.from_numpy(a) for a in data.collate(idx, pad_id))
                self.q.put(tuple(a.pin_memory() for a in t) if pin else t)
        except BaseException as e:
            self.q.put(e)
    def get(self):
        item = self.q.get()
        if isinstance(item, BaseException): raise RuntimeError("data-loader thread died") from item
        return item

# ------------------------------------------------------------------ prep mode
def run_prep():
    hf_login()
    build_data()
    print("PREP_DONE", flush=True)

# ------------------------------------------------------------------ train mode
CHAT_PY = '''"""Minimal chat loop:  python chat.py [path_to_this_folder]"""
import sys, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
path = sys.argv[1] if len(sys.argv) > 1 else "."
tok = AutoTokenizer.from_pretrained(path)
dev = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(path).to(dev)
if dev == "cuda": model = model.half()
model.eval()
history = []
print("Type 'quit' to exit, 'reset' to clear history.")
while True:
    user = input("You: ").strip()
    if user == "quit": break
    if user == "reset": history = []; continue
    history.append({"role": "user", "content": user})
    ids = tok.apply_chat_template(history, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(dev)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=256, do_sample=True, temperature=0.7, top_p=0.9,
                             repetition_penalty=1.1, pad_token_id=tok.pad_token_id)
    reply = tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    history.append({"role": "assistant", "content": reply})
    print("Bot:", reply)
'''

SAMPLE_PROMPTS = [
    [{"role": "user", "content": "Hi! How are you today?"}],
    [{"role": "user", "content": "Explain what photosynthesis is in two sentences."}],
    [{"role": "user", "content": "What is the capital of France?"}],
    [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": "Give me three tips for studying effectively."}],
]

def run_train():
    from transformers import LlamaForCausalLM, AutoModelForCausalLM, AutoTokenizer, GenerationConfig
    from torch.nn.parallel import DistributedDataParallel as DDP

    world = int(os.environ.get("WORLD_SIZE", 1)); local = int(os.environ.get("LOCAL_RANK", 0))
    cuda = torch.cuda.is_available()
    print(f"[rank {RANK}] worker started | pid {os.getpid()} | world={world} | cuda={cuda} | "
          f"gpus={torch.cuda.device_count() if cuda else 0}", flush=True)
    if cuda: torch.cuda.set_device(local)
    device = f"cuda:{local}" if cuda else "cpu"
    if world > 1:
        t_init = time.time()
        dist.init_process_group("nccl" if cuda else "gloo", timeout=timedelta(minutes=COMM_TIMEOUT_MIN))
        probe = torch.ones(1, device=device); dist.all_reduce(probe)
        assert int(probe.item()) == world
        log(f"Process group ready, all_reduce probe OK ({time.time()-t_init:.1f}s)")
    accum = max(1, round(GLOBAL_BATCH_TOKENS / (TOKENS_PER_MB * world)))

    tok = AutoTokenizer.from_pretrained(str(CHAT_TOK))
    BOS_ID, PAD_ID, IM_END_ID = (tok.convert_tokens_to_ids(t) for t in (BOS, PAD, IM_END))
    data = ConvData()
    log(f"Data: {len(data.train_idx):,} train convs ({data.train_tokens/1e6:.1f}M tokens) | {data.n_val:,} val convs")

    # ---- model: pretrained weights + 2 new chat tokens
    pre = find_pretrained(); log("Loading pretrained model from", pre)
    try:    raw = LlamaForCausalLM.from_pretrained(pre, attn_implementation="sdpa")
    except TypeError: raw = LlamaForCausalLM.from_pretrained(pre)
    raw = raw.float()
    torch.manual_seed(SEED)
    old_n = raw.get_input_embeddings().weight.shape[0]
    new_n = ((len(tok) + 63) // 64) * 64
    if new_n != old_n:
        raw.resize_token_embeddings(new_n)
        with torch.no_grad():                         # new rows = mean of existing rows + tiny noise
            w = raw.get_input_embeddings().weight
            w[old_n:] = w[:old_n].mean(dim=0, keepdim=True) + 0.01 * torch.randn_like(w[old_n:])
            ow = raw.get_output_embeddings().weight
            if ow.data_ptr() != w.data_ptr(): ow[old_n:] = w[old_n:]
    raw.config.use_cache = False
    raw.config.bos_token_id, raw.config.eos_token_id, raw.config.pad_token_id = BOS_ID, IM_END_ID, PAD_ID
    raw.to(device)
    n_params = sum(p.numel() for p in raw.parameters())
    tok_per_step = TOKENS_PER_MB * accum * world       # padded-token budget per optimizer step
    log(f"Model: {n_params/1e6:.1f}M params | embeddings {old_n} -> {new_n} | micro-batch budget {TOKENS_PER_MB} tok x accum {accum} x {world} GPU(s)")
    model = (DDP(raw, device_ids=[local], broadcast_buffers=False) if cuda else DDP(raw, broadcast_buffers=False)) if world > 1 else raw

    decay = [p for p in raw.parameters() if p.dim() >= 2]
    no_decay = [p for p in raw.parameters() if p.dim() < 2]
    kw = dict(lr=PEAK_LR, betas=BETAS, eps=1e-8)
    if cuda: kw["fused"] = True
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": WEIGHT_DECAY},
                             {"params": no_decay, "weight_decay": 0.0}], **kw)
    try:    scaler = torch.amp.GradScaler("cuda", enabled=cuda)
    except Exception: scaler = torch.cuda.amp.GradScaler(enabled=cuda)

    # ---- resume
    step, tokens_seen, elapsed_prev = 0, 0, 0.0
    ck_path = CKPT / "state.pt"
    if ck_path.exists():
        st = torch.load(ck_path, map_location="cpu", weights_only=False)
        raw.load_state_dict(st["model"]); opt.load_state_dict(st["opt"]); scaler.load_state_dict(st["scaler"])
        step, tokens_seen, elapsed_prev = st["step"], st["tokens_seen"], st["elapsed"]
        log(f"RESUMED from step {step} ({elapsed_prev/3600:.2f}h of training already done)")
    CKPT.mkdir(parents=True, exist_ok=True)

    wall_deadline = WALL_LIMIT_H * 3600 - FINAL_RESERVE_MIN * 60
    TRAIN_SECONDS = TRAIN_SECONDS_OVERRIDE or (elapsed_prev + (wall_deadline - wall_elapsed()))
    log(f"Training budget: {TRAIN_SECONDS/3600:.2f}h (or {MAX_EPOCHS:g} epochs, whichever ends first)")

    def lr_at(progress, step):
        if step < WARMUP_STEPS: return PEAK_LR * (step + 1) / WARMUP_STEPS
        p = min(1.0, progress)
        return PEAK_LR * (MIN_LR_FRAC + (1 - MIN_LR_FRAC) * 0.5 * (1 + math.cos(math.pi * p)))

    loader = Loader(data, SEED + 1000 * RANK + step, TOKENS_PER_MB, PAD_ID, device_index=local if cuda else None)
    # fixed validation micro-batches (each rank evaluates a disjoint share)
    vidx = data.val_idx[np.argsort(data.lens[data.val_idx], kind="stable")]
    vb = data.make_batches(vidx, TOKENS_PER_MB)
    sel = np.random.default_rng(0).permutation(len(vb))[:EVAL_BATCHES * world]
    val_batches = [tuple(torch.from_numpy(a) for a in data.collate(vb[k], PAD_ID)) for k in sel[RANK::world]]
    amp = lambda: torch.autocast("cuda" if cuda else "cpu", dtype=torch.float16, enabled=cuda)

    @torch.no_grad()
    def evaluate():
        raw.eval(); acc = torch.zeros(2, device=device, dtype=torch.float64)
        for x, y, am in val_batches:
            x, y, am = x.to(device), y.to(device), am.to(device)
            with amp(): l = raw(input_ids=x, attention_mask=am, labels=y).loss.float()
            n = (y[:, 1:] != -100).sum()
            acc[0] += l.double() * n; acc[1] += n
        if world > 1: dist.all_reduce(acc)
        raw.train(); return (acc[0] / acc[1].clamp(min=1)).item()

    @torch.no_grad()
    def sample(max_new=100):
        if not IS_MAIN: return
        try:
            raw.eval()
            for msgs in SAMPLE_PROMPTS[:3]:
                ids = tok(render(msgs, add_generation_prompt=True)[0], add_special_tokens=False, return_tensors="pt")["input_ids"].to(device)
                with amp():
                    out = raw.generate(input_ids=ids, max_new_tokens=max_new, do_sample=True, temperature=0.7, top_p=0.9,
                                       repetition_penalty=1.1, eos_token_id=IM_END_ID, pad_token_id=PAD_ID, use_cache=True)
                print(f"   [sample] {msgs[-1]['content']!r} -> {tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()!r}", flush=True)
        except Exception as e:
            print(f"   [sample] skipped ({type(e).__name__}: {str(e)[:100]})", flush=True)
        finally:
            raw.train()

    def save_ckpt(elapsed):
        tmp = ck_path.with_suffix(".tmp")
        torch.save({"model": raw.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                    "step": step, "tokens_seen": tokens_seen, "elapsed": elapsed}, tmp)
        os.replace(tmp, ck_path)

    base_val = evaluate()
    log(f"Baseline val loss on chat data (before SFT): {base_val:.4f}")

    # ---- train loop (rank 0 owns the clock; decisions are broadcast)
    raw.train()
    sess_start, last_ckpt = time.time(), time.time()
    elapsed_now = lambda: elapsed_prev + (time.time() - sess_start)
    t_log, tok_log, real_local, hist = time.time(), 0, 0, []
    loss_acc = torch.zeros((), device=device)
    first_step_pending = True
    log("Entering training loop ...")

    while True:
        epochs_done = tokens_seen / max(1, data.train_tokens)
        ctl = torch.tensor([elapsed_now(), float(time.time() - last_ckpt > CKPT_EVERY_MIN * 60),
                            float(wall_elapsed() >= wall_deadline), epochs_done], device=device, dtype=torch.float64)
        if world > 1: dist.broadcast(ctl, 0)
        el, do_ckpt, wall_stop, ep = ctl.tolist()
        prog = max(el / TRAIN_SECONDS, ep / MAX_EPOCHS)
        if prog >= 1.0 or wall_stop: break

        lr = lr_at(prog, step)
        for g in opt.param_groups: g["lr"] = lr

        step_real = 0
        for micro in range(accum):
            xc, yc, amc = loader.get()
            step_real += int(amc.sum())
            x, y, am = xc.to(device, non_blocking=True), yc.to(device, non_blocking=True), amc.to(device, non_blocking=True)
            sync = contextlib.nullcontext() if (world == 1 or micro == accum - 1) else model.no_sync()
            with sync:
                with amp(): loss = model(input_ids=x, attention_mask=am, labels=y).loss / accum
                scaler.scale(loss).backward()
            loss_acc += loss.detach().float()
        scaler.unscale_(opt)
        gnorm = torch.nn.utils.clip_grad_norm_(raw.parameters(), CLIP)
        scaler.step(opt); scaler.update()
        opt.zero_grad(set_to_none=True)

        step += 1
        real_local += step_real; tok_log += step_real
        tokens_seen += step_real * world                 # global estimate (rank 0's view x world)

        if first_step_pending:
            first_step_pending = False
            if cuda: torch.cuda.synchronize()
            mem = f" | peak GPU mem {torch.cuda.max_memory_allocated()/2**30:.1f} GiB" if cuda else ""
            log(f"FIRST_STEP_OK | step {step} | loss {loss_acc.item():.3f}{mem}")

        if step % LOG_EVERY == 0:
            if world > 1: dist.all_reduce(loss_acc); loss_acc /= world
            l = loss_acc.item() / LOG_EVERY; loss_acc.zero_()
            dt = time.time() - t_log
            log(f"step {step:6d} | loss {l:.4f} | lr {lr:.2e} | gnorm {float(gnorm):.2f} | "
                f"{tok_log*world/dt/1e3:5.1f}k tok/s | epoch {epochs_done:.2f}/{MAX_EPOCHS:g} | "
                f"train {el/3600:.2f}h / {TRAIN_SECONDS/3600:.2f}h")
            t_log, tok_log = time.time(), 0
        if step % EVAL_EVERY == 0:
            v = evaluate()
            hist.append({"step": step, "val_loss": v, "ppl": math.exp(min(v, 20)), "epoch": epochs_done})
            log(f"   >>> val loss {v:.4f} | ppl {math.exp(min(v, 20)):.2f}")
        if step % SAMPLE_EVERY == 0: sample()
        if do_ckpt:
            if IS_MAIN: save_ckpt(elapsed_now()); log("   checkpoint saved")
            last_ckpt = time.time()

    log(f"Training loop finished at step {step} | ~{tokens_seen/1e6:.0f}M real tokens | {tokens_seen/max(1, data.train_tokens):.2f} epochs")

    # ---- final eval + save (rank 0)
    final_val = evaluate()
    log(f"FINAL val loss {final_val:.4f} (baseline {base_val:.4f}) | ppl {math.exp(min(final_val, 20)):.2f}")
    if IS_MAIN:
        raw.config.use_cache = True
        OUT.mkdir(parents=True, exist_ok=True)
        gc = GenerationConfig(bos_token_id=BOS_ID, eos_token_id=[IM_END_ID, BOS_ID], pad_token_id=PAD_ID, do_sample=True,
                              temperature=0.7, top_p=0.9, repetition_penalty=1.1, max_new_tokens=256)
        raw.generation_config = gc
        raw.save_pretrained(OUT, safe_serialization=True)
        tok.save_pretrained(OUT)
        (OUT / "chat.py").write_text(CHAT_PY)
        (OUT / "README.md").write_text(
            "# Chat model (100M, trained from scratch + SFT)\n\nFormat: ChatML (`<|im_start|>role\\ncontent<|im_end|>`).\n\n"
            "```python\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n"
            "tok = AutoTokenizer.from_pretrained('PATH'); model = AutoModelForCausalLM.from_pretrained('PATH')\n"
            "ids = tok.apply_chat_template([{'role':'user','content':'Hello!'}], add_generation_prompt=True, return_tensors='pt', return_dict=True)\n"
            "out = model.generate(**ids, max_new_tokens=128)\nprint(tok.decode(out[0, ids['input_ids'].shape[1]:], skip_special_tokens=True))\n```\n"
            "Interactive: `python chat.py PATH`\n")
        (OUT / "training_summary.json").write_text(json.dumps({
            "params_M": n_params / 1e6, "steps": step, "real_tokens": tokens_seen, "epochs": tokens_seen / max(1, data.train_tokens),
            "baseline_val_loss": base_val, "final_val_loss": final_val, "train_hours": elapsed_now() / 3600,
            "eval_history": hist, "dataset": DATASET, "pretrained_from": pre, "world_size": world}, indent=2))
        log("Saved ->", OUT, sorted(p.name for p in OUT.iterdir()))

        # standalone reload test: weights + tokenizer + chat template + generation
        del opt, model
        tk2 = AutoTokenizer.from_pretrained(OUT)
        m2 = AutoModelForCausalLM.from_pretrained(OUT).to(device)
        if cuda: m2 = m2.half()
        m2.eval()
        probe = SAMPLE_PROMPTS[1]
        match = tk2.apply_chat_template(probe, tokenize=False, add_generation_prompt=True) == render(probe, True)[0]
        log(f"Chat template matches training format: {match}")
        print("\n--- SAMPLE CHATS ---", flush=True)
        for msgs in SAMPLE_PROMPTS:
            enc = tk2.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
            with torch.no_grad():
                out = m2.generate(**enc, max_new_tokens=120, pad_token_id=tk2.pad_token_id)
            print(f"User: {msgs[-1]['content']}\nBot : {tk2.decode(out[0, enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()}\n", flush=True)
        shutil.rmtree(CKPT, ignore_errors=True)
        log("ALL DONE. Total wall time: %.2fh" % (wall_elapsed() / 3600))
    if world > 1:
        dist.barrier(); dist.destroy_process_group()

def _hard_exit(code=0):
    if "ipykernel" in sys.modules: return
    sys.stdout.flush(); sys.stderr.flush()
    os._exit(code)

def main(mode=None):
    if "ipykernel" in sys.modules and mode is None:
        print("sft_pipeline.py: run `python sft_pipeline.py prep` then `torchrun ... sft_pipeline.py train` (see notebook)."); return
    mode = mode or (sys.argv[1] if len(sys.argv) > 1 else "train")
    if mode == "prep":
        run_prep(); _hard_exit(0)
    elif mode == "train":
        run_train(); _hard_exit(0)
    else:
        raise SystemExit(f"unknown mode {mode!r}")

if __name__ == "__main__":
    main()


## Launcher helper (streams output, watchdog, kills hangs)

In [ ]:
import os, sys, time, signal, subprocess, threading, queue
from pathlib import Path
import psutil

KERNEL_T0 = psutil.Process().create_time()          # session clock = kernel start
SCRIPT = "/kaggle/working/sft_pipeline.py"
RUN_LOG = "/kaggle/working/sft_run_log.txt"

def run_streamed(cmd, env_extra=None, ready_marker=None, startup_timeout=None,
                 stall_timeout=None, done_marker=None, done_grace=60):
    """Run cmd, stream its output into this cell, and kill it if it goes silent / hangs.
    Returns dict(rc, ready, done, reason)."""
    env = os.environ.copy()
    env.update(env_extra or {})
    env["PYTHONUNBUFFERED"] = "1"
    env["SESSION_T0"] = str(KERNEL_T0)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         bufsize=1, env=env, start_new_session=True)
    q = queue.Queue()
    def _reader():
        for line in p.stdout: q.put(line)
        q.put(None)
    threading.Thread(target=_reader, daemon=True).start()

    start = last = time.time()
    ready, done_at, reason, oom = ready_marker is None, None, None, False
    try:
      with open(RUN_LOG, "a") as lf:
        while True:
            try: line = q.get(timeout=5)
            except queue.Empty: line = ""
            now = time.time()
            if line is None: break                                   # child closed its output -> exited
            if line:
                print(line, end="", flush=True); lf.write(line); lf.flush(); last = now
                if not ready and ready_marker in line: ready = True
                if "OutOfMemoryError" in line or "CUDA out of memory" in line: oom = True
                if done_marker and done_marker in line and done_at is None: done_at = now
            if not ready and startup_timeout and now - start > startup_timeout:
                reason = f"no '{ready_marker}' within {startup_timeout}s"; break
            if ready and stall_timeout and now - last > stall_timeout:
                reason = f"no output for {stall_timeout}s (stalled)"; break
            if done_at and now - done_at > done_grace and p.poll() is None:
                reason = "finished but process did not exit"; break
    except BaseException:
        try: os.killpg(p.pid, signal.SIGKILL)
        except ProcessLookupError: pass
        raise
    if reason:
        print(f"\n[launcher] killing process group: {reason}", flush=True)
        try: os.killpg(p.pid, signal.SIGKILL)
        except ProcessLookupError: pass
    rc = p.wait()
    subprocess.run(["pkill", "-9", "-f", os.path.basename(SCRIPT)], check=False)   # leftovers holding GPU memory
    time.sleep(3)
    return {"rc": rc, "ready": ready, "done": done_at is not None, "reason": reason, "oom": oom}

## PREP - chat tokenizer + tokenized conversations (~10-15 min)

In [ ]:
# Chat tokenizer (+2 special tokens) and tokenized conversations from HuggingFaceTB/smol-smoltalk.
DATA_DIR = (Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/kaggle/working/data")) / f"sft_data{TAG}"
META = DATA_DIR / "meta.json"
ENV = {"PRETRAINED_DIR": PRETRAINED_DIR, "WALL_LIMIT_H": str(WALL_LIMIT_H), **EXTRA_ENV}
for attempt in range(1, 4):
    r = run_streamed([sys.executable, "-u", SCRIPT, "prep"], ENV, done_marker="PREP_DONE", done_grace=45, stall_timeout=900)
    if (r["done"] or r["rc"] == 0) and META.exists():
        print("\nPREP OK"); break
    print(f"\nPREP attempt {attempt} failed: {r}")
else:
    raise RuntimeError("PREP failed 3 times - see log above")

## TRAIN - DDP SFT on both T4s (self-healing)

In [ ]:
# Self-healing launcher (same as pretraining): OOM -> halve micro-batch budget; startup hang -> NCCL fallbacks -> 1 GPU;
# mid-run death -> resume from checkpoint. Global batch stays 32,768 tokens.
subprocess.run(["pkill", "-9", "-f", os.path.basename(SCRIPT)], check=False)
LADDER = [
    ("2xT4, NCCL default",           2, {}),
    ("2xT4, NCCL P2P off",           2, {"NCCL_P2P_DISABLE": "1"}),
    ("2xT4, NCCL P2P+SHM off",       2, {"NCCL_P2P_DISABLE": "1", "NCCL_SHM_DISABLE": "1", "NCCL_IB_DISABLE": "1"}),
    ("1xT4 fallback (same batch)",   1, {}),
]
idx, retries, budget, ok = 0, 0, 4096, False
while idx < len(LADDER) and not ok:
    if (time.time() - KERNEL_T0) / 3600 > WALL_LIMIT_H - 0.5:
        print("Not enough wall-clock budget left to start another attempt."); break
    name, nproc, extra = LADDER[idx]
    print(f"\n===== SFT attempt: {name} | micro-batch budget {budget} tokens =====", flush=True)
    cmd = [sys.executable, "-u", "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={nproc}", SCRIPT, "train"]
    env = {"NCCL_DEBUG": "WARN", "TOKENS_PER_MB": str(budget), "WALL_LIMIT_H": str(WALL_LIMIT_H),
           "PRETRAINED_DIR": PRETRAINED_DIR, "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True", **EXTRA_ENV, **extra}
    r = run_streamed(cmd, env, ready_marker="FIRST_STEP_OK", startup_timeout=600, stall_timeout=1500,
                     done_marker="ALL DONE", done_grace=180)
    if r["done"] or (r["rc"] == 0 and r["ready"]):
        ok = True
    elif r["oom"] and budget > 1024:
        budget //= 2; print(f"CUDA OOM -> retrying with micro-batch budget {budget}")
    elif r["ready"] and retries < 2:
        retries += 1; print(f"Training died mid-run ({r}); resuming from checkpoint (retry {retries}/2)")
    else:
        retries = 0; idx += 1; print(f"Attempt failed to start ({r}); trying next config")
print("\nSFT FINISHED OK" if ok else "\nSFT DID NOT COMPLETE - see log above")

## Try the chatbot

In [ ]:
# Chat with the fine-tuned model (loads from the saved folder, exactly how you would use it anywhere else).
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(CHAT_DIR)
dev = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(CHAT_DIR).to(dev)
if dev == "cuda": model = model.half()
model.eval()

def chat(user, history=None, system=None, max_new_tokens=200, temperature=0.7):
    msgs = ([{"role": "system", "content": system}] if system else []) + (history or []) + [{"role": "user", "content": user}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(dev)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature, top_p=0.9,
                             repetition_penalty=1.1, pad_token_id=tok.pad_token_id)
    reply = tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return reply, msgs[len(([{"role": "system"}] if system else [])):] + [{"role": "assistant", "content": reply}]

for q in ["Hello! Who are you?", "What is machine learning?", "Write a two-line poem about rain.", "How do I make a cup of tea?"]:
    print("User:", q); print("Bot :", chat(q)[0], "\n")

# Multi-turn example:
r1, hist = chat("Give me one tip for learning Python.")
r2, hist = chat("Can you explain it more simply?", history=hist)
print("Turn 1:", r1, "\nTurn 2:", r2)